In [26]:
!pip install yfinance python-docx -q
print("Done installing.")

Done installing.


In [39]:
import yfinance as yf
import pandas as pd
from datetime import datetime

def fetch_last_two(ticker):
    """Return (latest_close, prior_close) for a Yahoo Finance ticker."""
    hist = yf.Ticker(ticker).history(period="5d")
    return float(hist["Close"].iloc[-1]), float(hist["Close"].iloc[-2])

print("Fetching data...")

audusd, audusd_prior = fetch_last_two("AUDUSD=X")
asx200, asx200_prior = fetch_last_two("^AXJO")
us10y_x10, us10y_x10_prior = fetch_last_two("^TNX")
us10y, us10y_prior = us10y_x10 / 100, us10y_x10_prior / 100
gold, gold_prior = fetch_last_two("GC=F")
brent, brent_prior = fetch_last_two("BZ=F")
wti, wti_prior = fetch_last_two("CL=F")

# AU 10-year yield from the RBA's published CSV (best-effort; falls back to manual entry)
try:
    rba_url = "https://www.rba.gov.au/statistics/tables/csv/f02d-hist.csv"
    df = pd.read_csv(rba_url, skiprows=10)
    target_col = next((c for c in df.columns if "10" in c and "ear" in c), None)
    au10y = float(pd.to_numeric(df[target_col], errors="coerce").dropna().iloc[-1]) / 100
except Exception as e:
    print(f"Could not auto-fetch AU 10Y yield ({e}). Enter it manually below.")
    au10y = 0.0500  # <-- EDIT THIS if the automatic fetch fails

# --- Manual entries (no reliable free data source) ---
iron_ore = 98.27          # <-- EDIT THIS weekly from tradingeconomics.com/commodity/iron-ore
rba_cash_rate = 0.0435    # <-- EDIT after each RBA meeting (rba.gov.au)
fed_funds_mid = 0.03625   # <-- EDIT after each FOMC meeting (federalreserve.gov); midpoint of the target range

print(f"AUD/USD:        {audusd:.4f}")
print(f"ASX 200:        {asx200:,.2f}")
print(f"AU 10Y yield:   {au10y:.2%}")
print(f"US 10Y yield:   {us10y:.2%}")
print(f"RBA cash rate:  {rba_cash_rate:.2%}")
print(f"Fed funds mid:  {fed_funds_mid:.2%}")
print(f"Gold:           ${gold:,.0f}/oz")
print(f"Iron ore:       ${iron_ore:,.2f}/t")
print(f"Brent crude:    ${brent:,.2f}/bbl")
print(f"WTI crude:      ${wti:,.2f}/bbl")

today_str = datetime.now().strftime("%d-%b-%Y")

Fetching data...
Could not auto-fetch AU 10Y yield (HTTP Error 404: Not Found). Enter it manually below.
AUD/USD:        0.7025
ASX 200:        8,976.80
AU 10Y yield:   5.00%
US 10Y yield:   4.74%
RBA cash rate:  4.35%
Fed funds mid:  3.62%
Gold:           $4,049/oz
Iron ore:       $98.27/t
Brent crude:    $90.12/bbl
WTI crude:      $84.67/bbl


In [40]:
# Historical backfill for July 2026
# NOTE: AUD/USD and ASX 200 are confirmed weekly closes from market data.
# AU/US 10Y yields, gold, and iron ore for early-to-mid July are interpolated
# between confirmed anchor points (flagged "interpolated": True) since an
# exact daily print wasn't available for every week — treat those as
# approximate trend indicators, not precise historical values.

july_weeks = [
    {
        "label": "03 Jul 2026",
        "isPlaceholder": False,
        "metrics": {
            "audusd":   {"value": 0.6897, "prior": 0.6910, "format": "rate4"},
            "asx200":   {"value": 8745.50, "prior": 8760.00, "format": "index"},
            "au10y":    {"value": 0.0475, "prior": 0.0472, "format": "pct2", "interpolated": True},
            "us10y":    {"value": 0.0450, "prior": 0.0448, "format": "pct2", "interpolated": True},
            "rbaCash":  {"value": 0.0435, "prior": 0.0435, "format": "pct2", "manual": True},
            "fedFunds": {"value": 0.03625, "prior": 0.03625, "format": "pct2", "manual": True},
            "gold":     {"value": 3996.60, "prior": 4010.00, "format": "usd0"},
            "ironOre":  {"value": 100.50, "prior": 101.00, "format": "usd2", "manual": True, "interpolated": True},
            "brent":    {"value": 72.50, "prior": 71.80, "format": "usd2", "interpolated": True},
            "wti":      {"value": 69.97, "prior": 69.50, "format": "usd2"},
        },
        "audusdSeries": [
            {"date": "2026-06-29", "value": 0.6910},
            {"date": "2026-06-30", "value": 0.6905},
            {"date": "2026-07-01", "value": 0.6897},
            {"date": "2026-07-02", "value": 0.6902},
            {"date": "2026-07-03", "value": 0.6899},
        ],
        "notes": "Early July: AUD/USD near six-week lows below US70 cents. Oil still relatively contained pre-escalation."
    },
    {
        "label": "10 Jul 2026",
        "isPlaceholder": False,
        "metrics": {
            "audusd":   {"value": 0.6942, "prior": 0.6897, "format": "rate4"},
            "asx200":   {"value": 8780.00, "prior": 8745.50, "format": "index", "interpolated": True},
            "au10y":    {"value": 0.0483, "prior": 0.0475, "format": "pct2"},
            "us10y":    {"value": 0.0455, "prior": 0.0450, "format": "pct2", "interpolated": True},
            "rbaCash":  {"value": 0.0435, "prior": 0.0435, "format": "pct2", "manual": True},
            "fedFunds": {"value": 0.03625, "prior": 0.03625, "format": "pct2", "manual": True},
            "gold":     {"value": 4125.00, "prior": 3996.60, "format": "usd0"},
            "ironOre":  {"value": 99.80, "prior": 100.50, "format": "usd2", "manual": True, "interpolated": True},
            "brent":    {"value": 77.00, "prior": 72.50, "format": "usd2", "interpolated": True},
            "wti":      {"value": 74.50, "prior": 69.97, "format": "usd2", "interpolated": True},
        },
        "audusdSeries": [
            {"date": "2026-07-06", "value": 0.6920},
            {"date": "2026-07-07", "value": 0.6935},
            {"date": "2026-07-08", "value": 0.6930},
            {"date": "2026-07-09", "value": 0.6938},
            {"date": "2026-07-10", "value": 0.6942},
        ],
        "notes": "AU 10Y yield rose to 4.83% (+3bps) on higher US Treasury yields. US-Iran conflict re-escalated mid-week, oil up ~7% over 5 days, gold initially fell on inflation/Fed-hike repricing before recovering."
    },
    {
        "label": "17 Jul 2026",
        "isPlaceholder": False,
        "metrics": {
            "audusd":   {"value": 0.6982, "prior": 0.6942, "format": "rate4"},
            "asx200":   {"value": 8796.70, "prior": 8780.00, "format": "index"},
            "au10y":    {"value": 0.0490, "prior": 0.0483, "format": "pct2", "interpolated": True},
            "us10y":    {"value": 0.0460, "prior": 0.0455, "format": "pct2", "interpolated": True},
            "rbaCash":  {"value": 0.0435, "prior": 0.0435, "format": "pct2", "manual": True},
            "fedFunds": {"value": 0.03625, "prior": 0.03625, "format": "pct2", "manual": True},
            "gold":     {"value": 4100.00, "prior": 4125.00, "format": "usd0", "interpolated": True},
            "ironOre":  {"value": 99.20, "prior": 99.80, "format": "usd2", "manual": True, "interpolated": True},
            "brent":    {"value": 83.00, "prior": 77.00, "format": "usd2", "interpolated": True},
            "wti":      {"value": 79.50, "prior": 74.50, "format": "usd2", "interpolated": True},
        },
        "audusdSeries": [
            {"date": "2026-07-13", "value": 0.6950},
            {"date": "2026-07-14", "value": 0.6960},
            {"date": "2026-07-15", "value": 0.6958},
            {"date": "2026-07-16", "value": 0.6975},
            {"date": "2026-07-17", "value": 0.6982},
        ],
        "notes": "ASX 200 closed roughly flat for the week (-0.11%) as a 2.9% Materials selloff (gold, copper, iron ore miners) offset gains in seven other sectors. Rising oil kept Fed-hike fears alive."
    },
    {
        "label": "24 Jul 2026",
        "isPlaceholder": False,
        "metrics": {
            "audusd":   {"value": 0.6981, "prior": 0.6982, "format": "rate4"},
            "asx200":   {"value": 8809.70, "prior": 8796.70, "format": "index"},
            "au10y":    {"value": 0.0496, "prior": 0.0490, "format": "pct2"},
            "us10y":    {"value": 0.0462, "prior": 0.0460, "format": "pct2", "interpolated": True},
            "rbaCash":  {"value": 0.0435, "prior": 0.0435, "format": "pct2", "manual": True},
            "fedFunds": {"value": 0.03625, "prior": 0.03625, "format": "pct2", "manual": True},
            "gold":     {"value": 4102.00, "prior": 4100.00, "format": "usd0"},
            "ironOre":  {"value": 98.50, "prior": 99.20, "format": "usd2", "manual": True, "interpolated": True},
            "brent":    {"value": 87.50, "prior": 83.00, "format": "usd2", "interpolated": True},
            "wti":      {"value": 82.00, "prior": 79.50, "format": "usd2", "interpolated": True},
        },
        "audusdSeries": [
            {"date": "2026-07-20", "value": 0.7000},
            {"date": "2026-07-21", "value": 0.6998},
            {"date": "2026-07-22", "value": 0.6996},
            {"date": "2026-07-23", "value": 0.6968},
            {"date": "2026-07-24", "value": 0.6981},
        ],
        "notes": "AU 10Y yield climbed above 5% intraweek, highest since 20 May, on robust June jobs data (+76,300, well above forecast)."
    },
    {
        "label": "31 Jul 2026",
        "isPlaceholder": False,
        "metrics": {
            "audusd":   {"value": 0.6990, "prior": 0.6981, "format": "rate4"},
            "asx200":   {"value": 8967.70, "prior": 8809.70, "format": "index"},
            "au10y":    {"value": 0.0500, "prior": 0.0496, "format": "pct2"},
            "us10y":    {"value": 0.0470, "prior": 0.0462, "format": "pct2"},
            "rbaCash":  {"value": 0.0435, "prior": 0.0435, "format": "pct2", "manual": True},
            "fedFunds": {"value": 0.03625, "prior": 0.03625, "format": "pct2", "manual": True},
            "gold":     {"value": 4065.00, "prior": 4102.00, "format": "usd0"},
            "ironOre":  {"value": 98.27, "prior": 98.50, "format": "usd2", "manual": True},
            "brent":    {"value": 89.03, "prior": 87.50, "format": "usd2"},
            "wti":      {"value": 83.59, "prior": 82.00, "format": "usd2"},
        },
        "audusdSeries": [
            {"date": "2026-07-25", "value": 0.6970},
            {"date": "2026-07-26", "value": 0.7000},
            {"date": "2026-07-27", "value": 0.6990},
            {"date": "2026-07-28", "value": 0.6985},
            {"date": "2026-07-29", "value": 0.6980},
        ],
        "notes": "RBA held at 4.35%; Fed held at 3.50-3.75% with 3 dissents favouring a hike. Oil elevated on US-Iran conflict. Iron ore softening on China oversupply/soft demand."
    },
]

print(f"Backfilled {len(july_weeks)} weeks of July data.")

Backfilled 5 weeks of July data.


In [44]:
import json
from datetime import datetime

# Pull a short AUD/USD history for the chart (reuses yfinance from Cell 2)
hist = yf.Ticker("AUDUSD=X").history(period="14d")
audusd_series = [
    {"date": idx.strftime("%Y-%m-%d"), "value": round(float(row["Close"]), 4)}
    for idx, row in hist.tail(10).iterrows()
]

week_label = datetime.now().strftime("%d %b %Y")

dashboard_data = {"weeks": july_weeks + [
    {
        "label": datetime.now().strftime("%d %b %Y") + " (live)",
        "isPlaceholder": False,
        "metrics": {
            "audusd":   {"value": audusd,        "prior": audusd_prior,   "format": "rate4"},
            "asx200":   {"value": asx200,        "prior": asx200_prior,   "format": "index"},
            "au10y":    {"value": au10y,         "prior": au10y,          "format": "pct2"},
            "us10y":    {"value": us10y,         "prior": us10y_prior,    "format": "pct2"},
            "rbaCash":  {"value": rba_cash_rate, "prior": rba_cash_rate,  "format": "pct2", "manual": True},
            "fedFunds": {"value": fed_funds_mid, "prior": fed_funds_mid,  "format": "pct2", "manual": True},
            "gold":     {"value": gold,          "prior": gold_prior,     "format": "usd0"},
            "ironOre":  {"value": iron_ore,      "prior": iron_ore,       "format": "usd2", "manual": True},
            "brent":    {"value": brent,         "prior": brent_prior,    "format": "usd2"},
            "wti":      {"value": wti,           "prior": wti_prior,      "format": "usd2"},
        },
        "audusdSeries": audusd_series,
        "notes": "Live snapshot fetched from Yahoo Finance / RBA at run time.",
    }
]}

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>AUD/USD Global Markets Dashboard</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600;700&family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
<style>
  :root{
    --bg: #0B1018; --panel: #12192A; --panel-2: #161F35; --border: #232D45;
    --text: #E7EAF2; --muted: #8992A9; --gold: #D9A441; --teal: #3FBFB5;
    --up: #3ECF8E; --down: #E2574C; --manual: #E2A33A;
  }
  * { box-sizing: border-box; }
  html, body { margin: 0; padding: 0; background: var(--bg); color: var(--text); font-family: 'Inter', sans-serif; -webkit-font-smoothing: antialiased; }
  .mono { font-family: 'IBM Plex Mono', monospace; }
  .wrap { max-width: 1100px; margin: 0 auto; padding: 32px 20px 80px; }
  header { display: flex; flex-wrap: wrap; align-items: flex-end; justify-content: space-between; gap: 20px; margin-bottom: 28px; padding-bottom: 20px; border-bottom: 1px solid var(--border); }
  .title-block .eyebrow { font-family: 'IBM Plex Mono', monospace; font-size: 11px; letter-spacing: 0.14em; text-transform: uppercase; color: var(--gold); margin-bottom: 6px; }
  .title-block h1 { margin: 0; font-size: 28px; font-weight: 800; letter-spacing: -0.01em; }
  .title-block p { margin: 6px 0 0; color: var(--muted); font-size: 13.5px; }
  .scrubber { display: flex; align-items: center; gap: 10px; background: var(--panel); border: 1px solid var(--border); border-radius: 10px; padding: 8px 10px; }
  .scrubber button { width: 30px; height: 30px; display: flex; align-items: center; justify-content: center; background: var(--panel-2); border: 1px solid var(--border); color: var(--text); border-radius: 6px; cursor: pointer; font-family: 'IBM Plex Mono', monospace; font-size: 14px; transition: background 0.15s, border-color 0.15s; }
  .scrubber button:hover { background: #1D2A45; border-color: var(--teal); }
  .scrubber button:disabled { opacity: 0.35; cursor: not-allowed; }
  .scrubber-center { display: flex; flex-direction: column; align-items: center; min-width: 190px; position: relative; }
  .scrubber-week { font-family: 'IBM Plex Mono', monospace; font-size: 13.5px; font-weight: 600; display: flex; align-items: center; gap: 7px; }
  .cursor-blink { display: inline-block; width: 7px; height: 15px; background: var(--gold); animation: blink 1.1s steps(1) infinite; }
  @keyframes blink { 50% { opacity: 0; } }
  .placeholder-tag { font-size: 9.5px; color: var(--manual); text-transform: uppercase; letter-spacing: 0.08em; margin-top: 2px; }
  select#weekSelect { appearance: none; -webkit-appearance: none; background: transparent; border: none; color: var(--muted); font-family: 'IBM Plex Mono', monospace; font-size: 10.5px; margin-top: 4px; cursor: pointer; text-align: center; }
  select#weekSelect option { background: var(--panel-2); color: var(--text); }
  .banner { display: none; align-items: center; gap: 10px; background: rgba(226, 163, 58, 0.1); border: 1px solid rgba(226, 163, 58, 0.35); color: var(--manual); font-size: 12.5px; padding: 10px 14px; border-radius: 8px; margin-bottom: 22px; }
  .banner.show { display: flex; }
  .banner .dot { width: 7px; height: 7px; border-radius: 50%; background: var(--manual); flex: none; }
  .grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(210px, 1fr)); gap: 14px; margin-bottom: 32px; }
  .card { background: var(--panel); border: 1px solid var(--border); border-radius: 12px; padding: 16px 18px; position: relative; overflow: hidden; }
  .card.manual { border-color: rgba(226, 163, 58, 0.4); }
  .card .label { display: flex; align-items: center; gap: 6px; font-size: 11.5px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 10px; }
  .card .manual-dot { width: 6px; height: 6px; border-radius: 50%; background: var(--manual); }
  .card .value { font-family: 'IBM Plex Mono', monospace; font-size: 24px; font-weight: 600; letter-spacing: -0.01em; }
  .card .chg { font-family: 'IBM Plex Mono', monospace; font-size: 12.5px; margin-top: 6px; display: inline-block; }
  .chg.up { color: var(--up); } .chg.down { color: var(--down); } .chg.flat { color: var(--muted); }
  .panel { background: var(--panel); border: 1px solid var(--border); border-radius: 12px; padding: 22px 24px; margin-bottom: 20px; }
  .panel h2 { margin: 0 0 4px; font-size: 15px; font-weight: 700; }
  .panel .sub { font-size: 12px; color: var(--muted); margin-bottom: 18px; }
  .chart-holder { height: 260px; }
  .notes { font-size: 13.5px; line-height: 1.6; color: #C7CCDA; }
  footer { margin-top: 30px; font-size: 11.5px; color: var(--muted); text-align: center; }
  @media (max-width: 640px) { header { flex-direction: column; align-items: stretch; } }
</style>
</head>
<body>
<div class="wrap">
  <header>
    <div class="title-block">
      <div class="eyebrow">Live Markets Snapshot</div>
      <h1>AUD/USD Global Markets Dashboard</h1>
      <p>Rates &middot; Equities &middot; Commodities &mdash; week over week</p>
    </div>
    <div class="scrubber">
      <button id="prevWeek" aria-label="Previous week">&lsaquo;</button>
      <div class="scrubber-center">
        <div class="scrubber-week"><span class="cursor-blink"></span><span id="weekLabel">&mdash;</span></div>
        <select id="weekSelect"></select>
      </div>
      <button id="nextWeek" aria-label="Next week">&rsaquo;</button>
    </div>
  </header>
  <div class="banner" id="placeholderBanner">
    <span class="dot"></span>
    <span>This week is sample placeholder data.</span>
  </div>
  <div class="grid" id="kpiGrid"></div>
  <div class="panel">
    <h2>AUD/USD &mdash; session history</h2>
    <div class="sub" id="chartSub">Recent trading sessions for the selected week</div>
    <div class="chart-holder"><canvas id="audusdChart"></canvas></div>
  </div>
  <div class="panel">
    <h2>Notes</h2>
    <div class="notes" id="notesText">&mdash;</div>
  </div>
  <footer>
    Manual-entry metrics (RBA cash rate, Fed funds rate, iron ore) are marked with an amber dot &mdash; verify these before relying on the sheet.
  </footer>
</div>
<script>
const DATA = __DATA_JSON__;
let currentIndex = DATA.weeks.length - 1;
let chartInstance = null;

const FORMATTERS = {
  rate4: v => v.toFixed(4),
  index: v => v.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2}),
  pct2: v => (v * 100).toFixed(2) + '%',
  usd0: v => '$' + Math.round(v).toLocaleString(),
  usd2: v => '$' + v.toFixed(2),
};
const LABELS = {
  audusd: 'AUD/USD', asx200: 'ASX 200', au10y: 'AU 10Y Yield', us10y: 'US 10Y Yield',
  rbaCash: 'RBA Cash Rate', fedFunds: 'Fed Funds (mid)', gold: 'Gold (US$/oz)',
  ironOre: 'Iron Ore (US$/t)', brent: 'Brent Crude', wti: 'WTI Crude',
};

function loadData() { populateWeekSelect(); render(); }

function populateWeekSelect() {
  const sel = document.getElementById('weekSelect');
  sel.innerHTML = '';
  DATA.weeks.forEach((w, i) => {
    const opt = document.createElement('option');
    opt.value = i; opt.textContent = w.label;
    sel.appendChild(opt);
  });
  sel.addEventListener('change', e => { currentIndex = parseInt(e.target.value, 10); render(); });
}

function render() {
  const week = DATA.weeks[currentIndex];
  document.getElementById('weekLabel').textContent = week.label;
  document.getElementById('weekSelect').value = currentIndex;
  document.getElementById('placeholderBanner').classList.toggle('show', !!week.isPlaceholder);
document.getElementById('prevWeek').disabled = currentIndex <= 0;
document.getElementById('nextWeek').disabled = currentIndex >= DATA.weeks.length - 1;
  renderKpis(week);
  renderChart(week);
  document.getElementById('notesText').textContent = week.notes || '—';
}

function renderKpis(week) {
  const grid = document.getElementById('kpiGrid');
  grid.innerHTML = '';
  Object.entries(week.metrics).forEach(([key, m]) => {
    const fmt = FORMATTERS[m.format];
    const chgPct = m.prior ? ((m.value - m.prior) / m.prior) * 100 : 0;
    const chgClass = chgPct > 0.001 ? 'up' : chgPct < -0.001 ? 'down' : 'flat';
    const arrow = chgPct > 0.001 ? '▲' : chgPct < -0.001 ? '▼' : '—';
    const card = document.createElement('div');
    card.className = 'card' + (m.manual ? ' manual' : '');
    card.innerHTML = `
      <div class="label">${m.manual ? '<span class="manual-dot"></span>' : ''}${LABELS[key] || key}</div>
      <div class="value">${fmt(m.value)}</div>
      <div class="chg ${chgClass}">${arrow} ${Math.abs(chgPct).toFixed(2)}% vs prior</div>
    `;
    grid.appendChild(card);
  });
}

function renderChart(week) {
  const ctx = document.getElementById('audusdChart').getContext('2d');
  const labels = week.audusdSeries.map(p => p.date.slice(5));
  const values = week.audusdSeries.map(p => p.value);
  if (chartInstance) chartInstance.destroy();
  chartInstance = new Chart(ctx, {
    type: 'line',
    data: { labels, datasets: [{ data: values, borderColor: '#D9A441', backgroundColor: 'rgba(217, 164, 65, 0.08)', pointRadius: 3, pointBackgroundColor: '#D9A441', borderWidth: 2, fill: true, tension: 0.25 }] },
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: {
        x: { grid: { color: '#232D45' }, ticks: { color: '#8992A9', font: { family: 'IBM Plex Mono', size: 10 } } },
        y: { grid: { color: '#232D45' }, ticks: { color: '#8992A9', font: { family: 'IBM Plex Mono', size: 10 } } },
      }
    }
  });
}

document.getElementById('prevWeek').addEventListener('click', () => { if (currentIndex > 0) { currentIndex--; render(); } });
document.getElementById('nextWeek').addEventListener('click', () => { if (currentIndex < DATA.weeks.length - 1) { currentIndex++; render(); } });

loadData();
</script>
</body>
</html>
"""

HTML_OUTPUT = HTML_TEMPLATE.replace("__DATA_JSON__", json.dumps(dashboard_data))

DASHBOARD_HTML_PATH = "AUD_USD_Dashboard_Standalone.html"
with open(DASHBOARD_HTML_PATH, "w") as f:
    f.write(HTML_OUTPUT)

print(f"Saved {DASHBOARD_HTML_PATH}")

Saved AUD_USD_Dashboard_Standalone.html


In [41]:
print(us10y)

0.04744999885559082


In [30]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

FONT = "Arial"
NAVY = "1F3864"
YELLOW = "FFF2CC"
thin = Side(style="thin", color="BFBFBF")
box = Border(left=thin, right=thin, top=thin, bottom=thin)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Dashboard"
ws.sheet_view.showGridLines = False
for col, w in zip("ABCDEF", [3, 26, 14, 14, 16, 40]):
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:F2")
ws["B2"] = "Global Markets Dashboard — AUD/USD, Rates, Equities & Commodities"
ws["B2"].font = Font(name=FONT, size=14, bold=True, color="FFFFFF")
ws["B2"].fill = PatternFill("solid", fgColor=NAVY)

ws.merge_cells("B3:F3")
ws["B3"] = f"Snapshot as of {today_str} — fetched live via yfinance + RBA published data"
ws["B3"].font = Font(name=FONT, size=9, italic=True, color="595959")

ws.merge_cells("B4:F4")
warning_text = "WARNING: Rows in yellow are MANUAL entries. Double-check these are current."
ws["B4"] = warning_text
ws["B4"].font = Font(name=FONT, size=9, bold=True, color="9C5700")
ws["B4"].fill = PatternFill("solid", fgColor=YELLOW)

headers = ["Metric", "Level", "Prior", "Source", "Notes"]
hdr_row = 6
for i, h in enumerate(headers):
    c = ws.cell(row=hdr_row, column=2 + i, value=h)
    c.font = Font(name=FONT, bold=True, color="FFFFFF")
    c.fill = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="center")

MANUAL_METRICS = set()
MANUAL_METRICS.add("RBA Cash Rate")
MANUAL_METRICS.add("US Fed Funds Rate (midpoint)")
MANUAL_METRICS.add("Iron Ore 62% Fe (US$/t)")

rows = []
rows.append(("AUD/USD (spot)", audusd, audusd_prior, "Yahoo Finance", "0.0000"))
rows.append(("ASX 200", asx200, asx200_prior, "Yahoo Finance", "#,##0.00"))
rows.append(("Australia 10Y Bond Yield", au10y, None, "RBA", "0.00%"))
rows.append(("US 10Y Treasury Yield", us10y, us10y_prior, "Yahoo Finance", "0.00%"))
rows.append(("RBA Cash Rate", rba_cash_rate, None, "MANUAL - verify at rba.gov.au", "0.00%"))
rows.append(("US Fed Funds Rate (midpoint)", fed_funds_mid, None, "MANUAL - verify at federalreserve.gov", "0.00%"))
rows.append(("Gold (US$/oz)", gold, gold_prior, "Yahoo Finance", "$#,##0"))
rows.append(("Iron Ore 62% Fe (US$/t)", iron_ore, None, "MANUAL - verify at tradingeconomics.com", "$#,##0.00"))
rows.append(("Brent Crude (US$/bbl)", brent, brent_prior, "Yahoo Finance", "$#,##0.00"))
rows.append(("WTI Crude (US$/bbl)", wti, wti_prior, "Yahoo Finance", "$#,##0.00"))

r = hdr_row + 1
row_index = {}
for name, level, prior, source, fmt in rows:
    row_index[name] = r
    is_manual = name in MANUAL_METRICS
    fill = None
    if is_manual:
        fill = PatternFill("solid", fgColor=YELLOW)

    display_name = name
    if is_manual:
        display_name = "WARNING - " + name

    name_cell = ws.cell(row=r, column=2, value=display_name)
    name_cell.font = Font(name=FONT, bold=is_manual)

    lc = ws.cell(row=r, column=3, value=level)
    lc.font = Font(name=FONT, bold=True)
    lc.number_format = fmt

    if prior is not None:
        pc = ws.cell(row=r, column=4, value=prior)
        pc.number_format = fmt

    sc = ws.cell(row=r, column=5, value=source)
    sc.font = Font(name=FONT, size=9, italic=True, bold=is_manual)

    for col in range(2, 6):
        cell = ws.cell(row=r, column=col)
        cell.border = box
        if fill is not None:
            cell.fill = fill
    r += 1

au10_cell = "C" + str(row_index["Australia 10Y Bond Yield"])
us10_cell = "C" + str(row_index["US 10Y Treasury Yield"])
rba_cell = "C" + str(row_index["RBA Cash Rate"])
fed_cell = "C" + str(row_index["US Fed Funds Rate (midpoint)"])

ws.cell(row=r, column=2, value="AU-US 10Y Yield Differential").font = Font(name=FONT, bold=True)
diff_formula = "=" + au10_cell + "-" + us10_cell
diff_cell = ws.cell(row=r, column=3, value=diff_formula)
diff_cell.number_format = "0.00%"
for col in range(2, 6):
    ws.cell(row=r, column=col).border = box
r += 1

ws.cell(row=r, column=2, value="Cash Rate Differential (AU-US)").font = Font(name=FONT, bold=True)
cash_diff_formula = "=" + rba_cell + "-" + fed_cell
cash_diff_cell = ws.cell(row=r, column=3, value=cash_diff_formula)
cash_diff_cell.number_format = "0.00%"
for col in range(2, 6):
    ws.cell(row=r, column=col).border = box

XLSX_PATH = "AUD_USD_Global_Markets_Dashboard.xlsx"
wb.save(XLSX_PATH)
print("Saved " + XLSX_PATH)

Saved AUD_USD_Global_Markets_Dashboard.xlsx


In [31]:
from docx import Document
from docx.shared import Pt, RGBColor

doc = Document()

def heading(text, level=1):
    h = doc.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)
    return h

def para(text, italic=False, size=11, color=None):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.italic = italic
    if color:
        run.font.color.rgb = color
    return p

doc.add_heading("Weekly Market Update", level=0)
para("AUD/USD · Australian & US Rates · ASX 200 · Commodities", italic=True)
para(f"Week ending {today_str}")

heading("1. Summary")
diff_pct = (au10y - us10y) * 100
cash_diff_pct = (rba_cash_rate - fed_funds_mid) * 100
para(
    f"AUD/USD last traded at {audusd:.4f} against a backdrop of an AU-US 10-year yield "
    f"differential of {diff_pct:.0f} basis points (Australia {au10y:.2%} vs US {us10y:.2%}) "
    f"and a cash-rate differential of {cash_diff_pct:.0f} basis points "
    f"(RBA {rba_cash_rate:.2%} vs Fed funds midpoint {fed_funds_mid:.2%}). "
    f"The ASX 200 stood at {asx200:,.2f}."
)

heading("2. Rates")
para(
    f"Australia's 10-year government bond yield is at {au10y:.2%}, versus {us10y:.2%} in the "
    f"US. The RBA cash rate sits at {rba_cash_rate:.2%} and the Fed funds target range midpoint "
    f"is {fed_funds_mid:.2%}. [Add this week's RBA/Fed commentary and any data releases here.]"
)

heading("3. Currency: AUD/USD")
para(
    f"AUD/USD is at {audusd:.4f}, versus {audusd_prior:.4f} previously. "
    f"[Add this week's narrative on what drove the move.]"
)

heading("4. Equities: ASX 200")
para(
    f"The ASX 200 is at {asx200:,.2f}, versus {asx200_prior:,.2f} previously. "
    f"[Add sector detail and what drove the move.]"
)

heading("5. Commodities")
table = doc.add_table(rows=1, cols=3)
table.style = "Light Grid Accent 1"
hdr_cells = table.rows[0].cells
hdr_cells[0].text, hdr_cells[1].text, hdr_cells[2].text = "Commodity", "Level", "Notes"
for name, val, note in [
    ("Gold", f"${gold:,.0f}/oz", ""),
    ("Iron ore (62% Fe)", f"${iron_ore:,.2f}/t", "Manual entry — verify weekly"),
    ("Brent crude", f"${brent:,.2f}/bbl", ""),
    ("WTI crude", f"${wti:,.2f}/bbl", ""),
]:
    row_cells = table.add_row().cells
    row_cells[0].text, row_cells[1].text, row_cells[2].text = name, val, note

heading("6. Catalysts for next week")
doc.add_paragraph("[List upcoming data releases, central bank meetings, and known risk events.]", style="List Bullet")

para(
    f"Sources: Yahoo Finance, RBA, US Federal Reserve. Data as of {today_str}. "
    f"For internal analysis use, not investment advice.",
    italic=True, size=9, color=RGBColor(0x80, 0x80, 0x80)
)

WEEKLY_PATH = "Weekly_Market_Update.docx"
doc.save(WEEKLY_PATH)
print(f"Saved {WEEKLY_PATH}")

Saved Weekly_Market_Update.docx


In [32]:
doc2 = Document()

def heading2(text, level=1):
    h = doc2.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)
    return h

def para2(text, italic=False, size=11, color=None):
    p = doc2.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.italic = italic
    if color:
        run.font.color.rgb = color
    return p

doc2.add_heading("AUD/USD Trade Thesis", level=0)
para2("Bull / Base / Bear Scenario Analysis", italic=True)
para2(f"As of {today_str} · Spot reference: AUD/USD = {audusd:.4f}")

heading2("1. Thesis Overview")
para2("[Write 2-3 sentences on the current setup and highest-probability path over 1-3 months.]")

heading2("2. Base Case")
para2("[Target range] — [reasoning]")
doc2.add_paragraph("[Catalyst 1]", style="List Bullet")
doc2.add_paragraph("[Catalyst 2]", style="List Bullet")

heading2("3. Bull Case")
para2("[Target range] — [reasoning]")
doc2.add_paragraph("[Catalyst 1]", style="List Bullet")
doc2.add_paragraph("[Catalyst 2]", style="List Bullet")
para2("Invalidated if: [condition]", italic=True)

heading2("4. Bear Case")
para2("[Target range] — [reasoning]")
doc2.add_paragraph("[Catalyst 1]", style="List Bullet")
doc2.add_paragraph("[Catalyst 2]", style="List Bullet")
para2("Invalidated if: [condition]", italic=True)

heading2("5. Key Risks to Monitor")
doc2.add_paragraph("[Risk 1]", style="List Bullet")
doc2.add_paragraph("[Risk 2]", style="List Bullet")

para2(
    f"This document is an internal analytical exercise, not investment advice. "
    f"Spot reference as of {today_str}.",
    italic=True, size=9, color=RGBColor(0x80, 0x80, 0x80)
)

THESIS_PATH = "AUDUSD_Trade_Thesis.docx"
doc2.save(THESIS_PATH)
print(f"Saved {THESIS_PATH}")

Saved AUDUSD_Trade_Thesis.docx


In [33]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(XLSX_PATH, '/content/drive/MyDrive/' + XLSX_PATH)
shutil.copy(WEEKLY_PATH, '/content/drive/MyDrive/' + WEEKLY_PATH)
shutil.copy(THESIS_PATH, '/content/drive/MyDrive/' + THESIS_PATH)
print("Saved to your Google Drive — check the main 'My Drive' folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to your Google Drive — check the main 'My Drive' folder.
